# Ablation Training — Show, Attend and Tell

Trains **all 9 ablation variants**, evaluates each with the full beam × length-norm sweep,
and produces comparison visualisations against the baseline and paper.

| Group | Experiments |
|---|---|
| Architectural | `no_attention`, `no_beta_gate`, `feature_grid_7x7` |
| Regularisation | `lambda_0`, `lambda_0_5`, `lambda_2_0` |
| Fine-tune schedule | `finetune_ep3`, `finetune_ep10` |
| Combined | `lambda_0_5_finetune3` |

**Before running:**
- `Runtime → Change runtime type → GPU`
- Each experiment takes ~as long as the baseline run. Budget accordingly.
- Use `--resume` (already set below) to restart safely after a Colab disconnect.

**To run everything: `Runtime → Run all`**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path

# ── Edit if your folder is in a different location ───────────────────────────
REPO_DIR   = Path('/content/drive/MyDrive/Neural_Image_Caption_Generator')
DATA_ROOT  = REPO_DIR / 'data' / 'flickr8k'
VOCAB_PATH = DATA_ROOT / 'vocab.json'
BASE_OUT   = REPO_DIR / 'results' / 'ablation_results'   # per-experiment outputs
VIZ_OUT    = REPO_DIR / 'results' / 'ablation_experiments'  # plots + summary JSONs
# ─────────────────────────────────────────────────────────────────────────────

assert REPO_DIR.exists(), f'Repo not found: {REPO_DIR}'
BASE_OUT.mkdir(parents=True, exist_ok=True)
VIZ_OUT.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repo:      ', REPO_DIR)
print('Outputs:   ', BASE_OUT)
print('Viz/plots: ', VIZ_OUT)

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
!pytest -q

## 2. Configure Which Experiments to Run

Leave `RUN_ONLY = []` to train all 9 ablations, or list specific names to run a subset.
`RESUME = True` skips any experiment whose `best.pt` already exists — safe to re-run after a disconnect.

In [ ]:
# ── Leave empty to run all 9 ablations ──────────────────────────────────────
RUN_ONLY = []
# Example: RUN_ONLY = ['no_attention', 'no_beta_gate']

RESUME = True   # skip experiments whose best.pt already exists
# ─────────────────────────────────────────────────────────────────────────────

# Print the plan before committing
only_args = ['--only'] + RUN_ONLY if RUN_ONLY else []
!python scripts/run_ablations.py \
  --data_root "{DATA_ROOT}" \
  --vocab     "{VOCAB_PATH}" \
  {' '.join(only_args)}

## 3. Train All Ablations

Trains each variant then immediately evaluates it with 5 decoding configs
(beam 1/3/5 × length-normalisation on/off).

Output per experiment:
```
results/ablation_results/<name>/
    config.json
    training_log.csv
    checkpoints/best.pt
    checkpoints/epoch_NNN.pt
    eval_beam1_lnFalse.json
    eval_beam3_lnFalse.json
    eval_beam5_lnFalse.json
    eval_beam3_lnTrue.json
    eval_beam5_lnTrue.json
```

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'scripts/run_ablations.py', '--run',
    '--data_root', str(DATA_ROOT),
    '--vocab',     str(VOCAB_PATH),
]
if RESUME:   cmd.append('--resume')
if RUN_ONLY: cmd += ['--only'] + RUN_ONLY

print('Command:', ' '.join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    print('\n⚠ One or more experiments failed — check output above.')
else:
    print('\n✓ All experiments completed.')

## 4. Results Tables

In [ ]:
!python scripts/run_ablations.py --summary \
  --data_root "{DATA_ROOT}" \
  --vocab     "{VOCAB_PATH}"

In [ ]:
import json
import numpy as np

METRICS = ['bleu1', 'bleu2', 'bleu3', 'bleu4', 'meteor']
MNAMES  = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR']
PAPER   = [67.0, 44.8, 29.9, 19.5, 18.93]

EXPERIMENTS = [
    ('no_attention',          'architectural'),
    ('no_beta_gate',          'architectural'),
    ('feature_grid_7x7',      'architectural'),
    ('lambda_0',              'regularisation'),
    ('lambda_0_5',            'regularisation'),
    ('lambda_2_0',            'regularisation'),
    ('finetune_ep3',          'finetune_schedule'),
    ('finetune_ep10',         'finetune_schedule'),
    ('lambda_0_5_finetune3',  'combined'),
]

SWEEP_TAGS = [
    'beam1_lnFalse',
    'beam3_lnFalse',
    'beam5_lnFalse',
    'beam3_lnTrue',
    'beam5_lnTrue',
]

# Load all available results
all_results = {}   # {name: {sweep_tag: scores_dict}}
for name, _ in EXPERIMENTS:
    exp_dir = BASE_OUT / name
    sweep = {}
    for tag in SWEEP_TAGS:
        p = exp_dir / f'eval_{tag}.json'
        if p.exists():
            with open(p) as f:
                sweep[tag] = json.load(f)['scores']
    if sweep:
        all_results[name] = sweep

print(f'Loaded results for {len(all_results)}/{len(EXPERIMENTS)} experiments')
for name in all_results:
    print(f'  {name}: {len(all_results[name])} sweep configs')

## 5. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# For each experiment pick the sweep config with highest BLEU-4
def best_scores(name):
    if name not in all_results: return None, None
    return max(all_results[name].items(), key=lambda kv: kv[1].get('bleu4', 0))

rows = []
row_labels = []
for name, group in EXPERIMENTS:
    tag, scores = best_scores(name)
    if scores is None: continue
    rows.append([scores.get(m, 0)*100 for m in METRICS])
    row_labels.append(f"{name}\n[{tag}]")

if not rows:
    print('No results to plot yet — run Section 3 first.')
else:
    matrix = np.array(rows)

    fig, ax = plt.subplots(figsize=(10, max(4, len(rows)*0.7 + 1.5)))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto',
                   vmin=min(PAPER)*0.7, vmax=max(PAPER)*1.15)

    ax.set_xticks(range(len(MNAMES))); ax.set_xticklabels(MNAMES, fontsize=10)
    ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_title('Ablation Results — Best Decoding Config per Experiment\n'
                 '(green = above paper baseline, red = below)', fontsize=11)

    for i, row in enumerate(rows):
        for j, (val, paper) in enumerate(zip(row, PAPER)):
            diff = val - paper
            color = 'white' if abs(val - matrix[:,j].mean()) > matrix[:,j].std() else 'black'
            ax.text(j, i, f'{val:.1f}\n({diff:+.1f})', ha='center', va='center',
                    fontsize=7, color=color)

    # Paper row as horizontal lines
    for j, p in enumerate(PAPER):
        ax.axvline(j - 0.5, color='white', linewidth=0.5)

    plt.colorbar(im, ax=ax, fraction=0.015, pad=0.02, label='Score')
    plt.tight_layout()
    out = VIZ_OUT / 'ablation_heatmap.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── BLEU-4 bar chart: best config per experiment vs paper ─────────────────────
if all_results:
    names_done = [n for n, _ in EXPERIMENTS if n in all_results]
    bleu4_vals = [best_scores(n)[1].get('bleu4', 0)*100 for n in names_done]
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(names_done)))

    fig, ax = plt.subplots(figsize=(max(10, len(names_done)*1.4), 5))
    bars = ax.bar(names_done, bleu4_vals, color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(PAPER[3], color='red', linestyle='--', linewidth=2,
               label=f'Paper baseline ({PAPER[3]})')

    for bar, val in zip(bars, bleu4_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('BLEU-4'); ax.set_title('BLEU-4 by Ablation (best decoding config)')
    ax.set_xticklabels(names_done, rotation=25, ha='right', fontsize=9)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out = VIZ_OUT / 'ablation_bleu4_bars.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── BLEU-4 across beam configs for each experiment ────────────────────────────
if all_results:
    beam_x = [1, 3, 5, 3, 5]   # beam width per sweep tag
    ln_marker = ['o', 's', '^', 's', '^']  # shape indicates length_norm
    ln_label  = [False, False, False, True, True]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: no length_norm configs
    # Right: with length_norm configs
    for ax_idx, use_ln in enumerate([False, True]):
        ax = axes[ax_idx]
        tags = [t for t, ln in zip(SWEEP_TAGS, ln_label) if ln == use_ln]
        bws  = [bx for bx, ln in zip(beam_x, ln_label) if ln == use_ln]

        # Paper reference line (flat — paper reports single greedy number)
        ax.axhline(PAPER[3], color='red', linestyle='--', linewidth=1.5,
                   label='Paper baseline', zorder=1)

        colors = plt.cm.tab10(np.linspace(0, 0.9, len(all_results)))
        for (name, _), color in zip(
                [(n,g) for n,g in EXPERIMENTS if n in all_results], colors):
            bleu4s = [all_results[name].get(t, {}).get('bleu4', 0)*100
                      for t in tags]
            ax.plot(bws, bleu4s, 'o-', color=color, label=name, linewidth=1.5)

        ln_str = 'with length normalisation' if use_ln else 'no length normalisation'
        ax.set_title(f'BLEU-4 vs Beam Width ({ln_str})')
        ax.set_xlabel('Beam width'); ax.set_ylabel('BLEU-4')
        ax.set_xticks(sorted(set(bws)))
        ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=0.3)

    plt.tight_layout()
    out = VIZ_OUT / 'ablation_beam_sweep_lines.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── All metrics: grouped bar chart, best config per experiment ────────────────
if all_results:
    names_done = [n for n, _ in EXPERIMENTS if n in all_results]
    n = len(names_done)
    x = np.arange(len(MNAMES))
    width = 0.75 / (n + 1)
    colors = plt.cm.tab10(np.linspace(0, 0.9, n))

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, (name, color) in enumerate(zip(names_done, colors)):
        _, scores = best_scores(name)
        vals = [scores.get(m, 0)*100 for m in METRICS]
        ax.bar(x + i*width - (n*width)/2 + width/2, vals, width,
               label=name, color=color, alpha=0.85)

    ax.plot(x, PAPER, 'rv-', markersize=8, linewidth=2,
            label='Paper Soft-Att (Table 1)', zorder=5)

    ax.set_xticks(x); ax.set_xticklabels(MNAMES, fontsize=11)
    ax.set_ylabel('Score'); ax.set_title('All Metrics: Ablations vs Paper Baseline')
    ax.legend(fontsize=7, loc='upper right', ncol=2); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out = VIZ_OUT / 'ablation_all_metrics.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── Radar chart: all ablations relative to paper = 1.0 ────────────────────────
if all_results:
    N = len(MNAMES)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

    def to_radar(scores):
        vals = [scores.get(m, 0)*100 / p for m, p in zip(METRICS, PAPER)]
        return vals + [vals[0]]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.plot(angles, [1.0]*N+[1.0], 'r--', linewidth=2, label='Paper baseline (1.0)')
    ax.fill(angles, [1.0]*N+[1.0], alpha=0.06, color='red')

    colors = plt.cm.tab10(np.linspace(0, 0.9, len(all_results)))
    for (name, _), color in zip(
            [(n,g) for n,g in EXPERIMENTS if n in all_results], colors):
        _, scores = best_scores(name)
        vals = to_radar(scores)
        ax.plot(angles, vals, 'o-', linewidth=1.5, color=color, label=name)
        ax.fill(angles, vals, alpha=0.06, color=color)

    ax.set_xticks(angles[:-1]); ax.set_xticklabels(MNAMES, fontsize=11)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([0.5, 1.0, 1.25])
    ax.set_yticklabels(['0.5×', '1.0× (paper)', '1.25×'], fontsize=8)
    ax.set_title('Ablations Relative to Paper Baseline', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), fontsize=8)
    plt.tight_layout()
    out = VIZ_OUT / 'ablation_radar.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

In [ ]:
# ── Training curves: val BLEU-4 per epoch for every completed experiment ──────
import csv

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 0.9, len(EXPERIMENTS)))
plotted = False

for (name, _), color in zip(EXPERIMENTS, colors):
    log = BASE_OUT / name / 'training_log.csv'
    if not log.exists(): continue
    eps, b4s = [], []
    with open(log) as f:
        for row in csv.DictReader(f):
            eps.append(int(row['epoch']))
            b4s.append(float(row['val_bleu4'])*100)
    ax.plot(eps, b4s, 'o-', color=color, label=name, linewidth=1.5, markersize=4)
    plotted = True

if plotted:
    ax.axhline(PAPER[3], color='red', linestyle='--', linewidth=2,
               label=f'Paper target ({PAPER[3]})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val BLEU-4')
    ax.set_title('Validation BLEU-4 During Training — All Ablations')
    ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    out = VIZ_OUT / 'ablation_training_curves.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')
else:
    print('No training logs found yet — run Section 3 first.')

## 6. Attention Map Comparison Across Ablations

Runs the same test images through each trained model and shows side-by-side
attention overlays. Only models with `best.pt` are included.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from torchvision import transforms
from PIL import Image as PILImage
from pathlib import Path

from config import ATTENTION_DIM, DECODER_DIM, EMBED_DIM, MAX_DECODE_LEN
from models import Encoder, Decoder
from utils import Vocabulary, greedy_decode, load_flickr8k_captions

_MEAN = np.array([0.485, 0.456, 0.406])
_STD  = np.array([0.229, 0.224, 0.225])
_TF   = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN.tolist(), _STD.tolist()),
])

def unnorm(t):
    return (np.clip(t.permute(1,2,0).numpy()*_STD+_MEAN, 0,1)*255).astype(np.uint8)

@torch.no_grad()
def load_model(ckpt_path, vocab, device, attention_mode='soft',
               use_beta_gate=True, feature_grid_size=14):
    ckpt = torch.load(ckpt_path, map_location=device)
    enc = Encoder(fine_tune=False, feature_grid_size=feature_grid_size).to(device)
    dec = Decoder(attention_dim=ATTENTION_DIM, embed_dim=EMBED_DIM,
                  decoder_dim=DECODER_DIM, vocab_size=len(vocab), dropout=0.0,
                  attention_mode=attention_mode, use_beta_gate=use_beta_gate).to(device)
    enc.load_state_dict(ckpt['encoder'])
    dec.load_state_dict(ckpt['decoder'])
    enc.eval(); dec.eval()
    return enc, dec

def overlay_alpha(img_np, alpha_1d, sigma=8.0):
    L = len(alpha_1d); g = int(round(L**0.5))
    up = F.interpolate(torch.tensor(alpha_1d).view(1,1,g,g),
                       (224,224), mode='bilinear', align_corners=False).squeeze().numpy()
    up = gaussian_filter(up, sigma)
    if up.max(): up /= up.max()
    gray = np.dot(img_np/255., [0.299,0.587,0.114])
    rgb = np.stack([gray]*3, -1) + up[:,:,None]*0.7
    return (np.clip(rgb,0,1)*255).astype(np.uint8)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab  = Vocabulary.load(str(VOCAB_PATH))
print('Helpers loaded.')

In [ ]:
# Ablation-specific model flags (must match training config)
MODEL_FLAGS = {
    'no_attention':     dict(attention_mode='none', use_beta_gate=True,  feature_grid_size=14),
    'no_beta_gate':     dict(attention_mode='soft', use_beta_gate=False, feature_grid_size=14),
    'feature_grid_7x7': dict(attention_mode='soft', use_beta_gate=True,  feature_grid_size=7),
    # All others use defaults
}

# Pick sample images
N_IMAGES = 3
img_to_caps = load_flickr8k_captions(str(DATA_ROOT), 'test')
sample_names   = sorted(img_to_caps.keys())[:N_IMAGES]
sample_tensors = [_TF(PILImage.open(DATA_ROOT/'images'/n).convert('RGB'))
                  for n in sample_names]

# Only include ablations that have a best.pt
ready = [(n,g) for n,g in EXPERIMENTS
         if (BASE_OUT / n / 'checkpoints' / 'best.pt').exists()]

if not ready:
    print('No trained checkpoints found yet — run Section 3 first.')
else:
    MAX_WORDS = 7
    for img_idx, (img_tensor, img_name) in enumerate(zip(sample_tensors, sample_names)):
        img_np = unnorm(img_tensor)
        n_rows = len(ready)
        n_cols = MAX_WORDS + 1

        fig, axes = plt.subplots(n_rows, n_cols,
                                  figsize=(2.2*n_cols, 2.6*n_rows))
        if n_rows == 1: axes = axes[np.newaxis, :]

        for row_idx, (name, group) in enumerate(ready):
            ckpt = BASE_OUT / name / 'checkpoints' / 'best.pt'
            flags = MODEL_FLAGS.get(name, dict(attention_mode='soft',
                                               use_beta_gate=True, feature_grid_size=14))
            enc, dec = load_model(str(ckpt), vocab, device, **flags)

            with torch.no_grad():
                caption, alphas, token_ids = greedy_decode(
                    enc, dec, img_tensor.unsqueeze(0).to(device),
                    vocab, device, max_len=MAX_DECODE_LEN)
            words = [vocab.idx2word.get(i,'<unk>') for i in token_ids]

            axes[row_idx, 0].imshow(img_np)
            axes[row_idx, 0].set_title(name, fontsize=6.5, fontweight='bold')
            axes[row_idx, 0].set_xlabel(f'"{caption[:45]}"', fontsize=5.5, labelpad=2)
            axes[row_idx, 0].axis('off')

            for w in range(MAX_WORDS):
                ax = axes[row_idx, w+1]
                if w < len(words) and w < alphas.shape[0]:
                    ax.imshow(overlay_alpha(img_np, alphas[w].numpy()))
                    ax.set_title(words[w], fontsize=8)
                ax.axis('off')

            del enc, dec
            torch.cuda.empty_cache()

        fig.suptitle(f'Attention — {Path(img_name).stem}', fontsize=10, y=1.01)
        plt.tight_layout()
        out = VIZ_OUT / f'attn_ablation_{img_idx+1:02d}.png'
        plt.savefig(str(out), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {out}')

## 7. Summary of All Saved Files

In [ ]:
print(f'Visualisations in {VIZ_OUT}:')
for f in sorted(VIZ_OUT.iterdir()):
    if f.is_file():
        print(f'  {f.name:<50}  {f.stat().st_size/1e3:>8.1f} KB')

print(f'\nPer-experiment outputs in {BASE_OUT}:')
for exp_d in sorted(BASE_OUT.iterdir()):
    if not exp_d.is_dir(): continue
    files = list(exp_d.rglob('*'))
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    ckpt_exists = (exp_d / 'checkpoints' / 'best.pt').exists()
    evals_done  = len(list(exp_d.glob('eval_*.json')))
    print(f'  {exp_d.name:<28}  best.pt={ckpt_exists}  evals={evals_done}/5  {total_mb:.0f} MB')